In [1]:
import snowflake.connector
import pandas as pd
import getpass
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

conn = snowflake.connector.connect(
    user=input("Snowflake username: "),
    password=getpass.getpass("Snowflake password: "),
    account=input("Account identifier: "),
    warehouse='CMS_WH',
    database='CMS_PROJECT',
    schema='REVENUE_CYCLE'
)

print("Connected!")


Snowflake username:  shantanunayar
Snowflake password:  ········
Account identifier:  QHJNHYN-WZ62559


Connected!


In [2]:
query = """
SELECT *
FROM MEDICARE_PROVIDER_SERVICE
SAMPLE (500000 ROWS)
"""

df = pd.read_sql(query, conn)
print(f"Rows loaded: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


/var/folders/10/cg68x6sj1rx_46lzjx372ycc0000gn/T/ipykernel_11721/3223539010.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Rows loaded: 500,000
Columns: 28


In [3]:
# 1. Raw leakage in dollars
df['LEAKAGE'] = df['AVG_SBMTD_CHRG'] - df['AVG_MDCR_PYMT_AMT']

# 2. Leakage rate as a percentage of submitted charge
df['LEAKAGE_RATE'] = df['LEAKAGE'] / df['AVG_SBMTD_CHRG']

# 3. Total dollar leakage weighted by volume
df['TOTAL_LEAKAGE'] = df['LEAKAGE'] * df['TOT_SRVCS']

# 4. Payment ratio - what fraction of submitted charge did Medicare actually pay
df['PAYMENT_RATIO'] = df['AVG_MDCR_PYMT_AMT'] / df['AVG_SBMTD_CHRG']

# 5. Allowed but unpaid - patient cost sharing amount
df['PATIENT_COST_SHARE'] = df['AVG_MDCR_ALOWD_AMT'] - df['AVG_MDCR_PYMT_AMT']

# Preview
df[['AVG_SBMTD_CHRG', 'AVG_MDCR_PYMT_AMT', 'LEAKAGE', 
    'LEAKAGE_RATE', 'TOTAL_LEAKAGE', 'PAYMENT_RATIO', 
    'PATIENT_COST_SHARE']].head(10)

,AVG_SBMTD_CHRG,AVG_MDCR_PYMT_AMT,LEAKAGE,LEAKAGE_RATE,TOTAL_LEAKAGE,PAYMENT_RATIO,PATIENT_COST_SHARE
0,297.5625,19.744375,277.818125,0.933646,4445.09,0.066354,6.634375
1,120.0000,55.152692,64.847308,0.540394,1686.03,0.459606,53.042692
2,550.0000,162.230938,387.769063,0.705035,12408.61,0.294965,61.122812
3,176.0000,51.590000,124.410000,0.706875,2737.02,0.293125,0.000000
4,269.0000,55.540000,213.460000,0.793532,2348.06,0.206468,48.669091
5,646.0000,118.753111,527.246889,0.816172,23726.11,0.183828,55.253111
6,22.0000,11.284137,10.715863,0.487085,16395.27,0.512915,2.937837
7,120.0000,37.699545,82.300455,0.685837,1810.61,0.314163,11.885000
8,430.0000,54.585156,375.414844,0.873058,108494.89,0.126942,17.748270
9,433.0000,72.511562,360.488438,0.832537,11535.63,0.167463,7.683750


In [4]:

threshold = df['LEAKAGE_RATE'].quantile(0.75)
print(f"Denial proxy threshold (75th percentile leakage rate): {threshold:.2%}")


df['DENIAL_PROXY'] = (df['LEAKAGE_RATE'] > threshold).astype(int)


print(f"\nClass distribution:")
print(df['DENIAL_PROXY'].value_counts())
print(f"\nDenial rate: {df['DENIAL_PROXY'].mean():.2%}")

Denial proxy threshold (75th percentile leakage rate): 83.11%

Class distribution:
DENIAL_PROXY
0    375000
1    125000
Name: count, dtype: int64

Denial rate: 25.00%


In [5]:
df['IS_FACILITY'] = (df['PLACE_OF_SRVC'] == 'F').astype(int)
df['IS_DRUG'] = (df['HCPCS_DRUG_IND'] == 'Y').astype(int)
df['IS_PARTICIPATING'] = (df['RNDRNG_PRVDR_MDCR_PRTCPTG_IND'] == 'Y').astype(int)
provider_freq = df['RNDRNG_PRVDR_TYPE'].value_counts(normalize=True)
df['PROVIDER_TYPE_FREQ'] = df['RNDRNG_PRVDR_TYPE'].map(provider_freq)
state_freq = df['RNDRNG_PRVDR_STATE_ABRVTN'].value_counts(normalize=True)
df['STATE_FREQ'] = df['RNDRNG_PRVDR_STATE_ABRVTN'].map(state_freq)

df[['PLACE_OF_SRVC','IS_FACILITY','HCPCS_DRUG_IND','IS_DRUG','RNDRNG_PRVDR_MDCR_PRTCPTG_IND','IS_PARTICIPATING','RNDRNG_PRVDR_TYPE','PROVIDER_TYPE_FREQ','RNDRNG_PRVDR_STATE_ABRVTN','STATE_FREQ']].head(10)

,PLACE_OF_SRVC,IS_FACILITY,HCPCS_DRUG_IND,IS_DRUG,RNDRNG_PRVDR_MDCR_PRTCPTG_IND,IS_PARTICIPATING,RNDRNG_PRVDR_TYPE,PROVIDER_TYPE_FREQ,RNDRNG_PRVDR_STATE_ABRVTN,STATE_FREQ
0,F,1,N,0,Y,1,Diagnostic Radiology,0.114700,MO,0.019048
1,O,0,N,0,Y,1,Physical Therapist in Private Practice,0.042380,CA,0.085270
2,O,0,N,0,Y,1,Interventional Pain Management,0.002806,AL,0.015906
3,O,0,N,0,Y,1,Internal Medicine,0.082100,CA,0.085270
4,O,0,N,0,Y,1,Internal Medicine,0.082100,NC,0.035380
5,O,0,N,0,Y,1,Urology,0.016690,CA,0.085270
6,O,0,N,0,Y,1,Allergy/ Immunology,0.002640,NV,0.007666
7,O,0,N,0,Y,1,Diagnostic Radiology,0.114700,AZ,0.022458
8,O,0,N,0,Y,1,Nurse Practitioner,0.099622,NY,0.063544
9,O,0,N,0,Y,1,Family Practice,0.078686,TX,0.070174


In [6]:
feature_cols = [
    'AVG_MDCR_PYMT_AMT',
    'AVG_MDCR_STDZD_AMT', 
    'LEAKAGE',
    'LEAKAGE_RATE',
    'TOTAL_LEAKAGE',
    'PAYMENT_RATIO',
    'PATIENT_COST_SHARE',
    'TOT_BENES',
    'TOT_SRVCS',
    'TOT_BENE_DAY_SRVCS',
    'IS_FACILITY',
    'IS_DRUG',
    'IS_PARTICIPATING',
    'PROVIDER_TYPE_FREQ',
    'STATE_FREQ'
]

target_col = 'DENIAL_PROXY'

print("Null counts in feature columns:")
print(df[feature_cols].isnull().sum())
print(f"\nNull count in target: {df[target_col].isnull().sum()}")

Null counts in feature columns:
AVG_MDCR_PYMT_AMT     0
AVG_MDCR_STDZD_AMT    0
LEAKAGE               0
LEAKAGE_RATE          0
TOTAL_LEAKAGE         0
PAYMENT_RATIO         0
PATIENT_COST_SHARE    0
TOT_BENES             0
TOT_SRVCS             0
TOT_BENE_DAY_SRVCS    0
IS_FACILITY           0
IS_DRUG               0
IS_PARTICIPATING      0
PROVIDER_TYPE_FREQ    0
STATE_FREQ            0
dtype: int64

Null count in target: 0
